<a href="https://colab.research.google.com/github/paulpdelancy-spec/gdp-dashboard/blob/main/Copy_of_Master_Cell_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── 0. INSTALLATION ───────────────────────────────────
!pip install -q -U "google-genai>=1.16.0" tqdm pytz beautifulsoup4

# ── 1. SYSTEM IMPORTS & CORE CONFIG ──────────────────
import os, secrets, string, datetime, time, pytz, smtplib, requests
from tqdm import tqdm
from bs4 import BeautifulSoup
from google.colab import auth, userdata
from google import genai
from google.genai import types
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# Auth & Credentials
auth.authenticate_user()
GEMINI_API_KEY  = userdata.get('GEMINI_API_KEY')
SENDER_PASSWORD = userdata.get('GMAIL_APP_PASSWORD')
SENDER_EMAIL    = "paul.pdelancy@gmail.com"
CONTACT_EMAILS  = ["paulacumberbatch@yahoo.com", "paul.pdelancy@gmail.com"]
DRIVE_FOLDER_ID = "1wKOXcL5T2KW4JeOkydB8AxOUuwo1auYI"

client = genai.Client(api_key=GEMINI_API_KEY)
OK_TZ = pytz.timezone('America/Chicago')

# ── 2. DATA REPOSITORIES (71 SITES) ──────────────────
SPORTS_SITES = [
    {"url": "https://www.afa.com.ar/es/", "name": "AFA Argentina"},
    {"url": "https://dimayor.com.co/", "name": "Dimayor Colombia"},
    {"url": "https://www.cfl.ca/", "name": "CFL Canada"},
    {"url": "https://www.mlb.com/", "name": "MLB USA"},
    {"url": "https://www.mlssoccer.com/", "name": "MLS USA"},
    {"url": "https://www.nba.com/", "name": "NBA USA"},
    {"url": "https://www.nfl.com/", "name": "NFL USA"},
    {"url": "https://www.nhl.com/", "name": "NHL USA/Canada"},
    {"url": "https://www.ligamx.net/", "name": "Liga MX Mexico"},
    {"url": "https://www.cbf.com.br/", "name": "CBF Brazil"},
    {"url": "https://www.premierleague.com/", "name": "Premier League UK"},
    {"url": "https://www.bundesliga.com/en/bundesliga/", "name": "Bundesliga Germany"},
    {"url": "https://www.bundesliga.at/de", "name": "Bundesliga Austria"},
    {"url": "https://www.laliga.com/en-GB", "name": "La Liga Spain"},
    {"url": "https://www.legaseriea.it/it", "name": "Serie A Italy"},
    {"url": "https://ligue1.com/", "name": "Ligue 1 France"},
    {"url": "https://top14.lnr.fr/", "name": "Top 14 Rugby France"},
    {"url": "https://www.efl.com/", "name": "EFL England"},
    {"url": "https://eredivisie.nl/", "name": "Eredivisie Netherlands"},
    {"url": "https://www.ligaportugal.pt/", "name": "Liga Portugal"},
    {"url": "https://sfl.ch/", "name": "Swiss Football League"},
    {"url": "https://www.tff.org/", "name": "TFF Turkey"},
    {"url": "https://www.afl.com.au/", "name": "AFL Australia"},
    {"url": "https://www.nrl.com/", "name": "NRL Australia"},
    {"url": "http://eng.koreabaseball.com/", "name": "KBO Korea"},
    {"url": "https://www.jleague.jp/en/", "name": "J-League Japan"},
    {"url": "https://www.iplt20.com/", "name": "IPL India"},
    {"url": "https://www.indiansuperleague.com/", "name": "ISL India"},
    {"url": "https://sports.sina.com.cn/csl/", "name": "CSL China"},
    {"url": "https://uscenterforsafesport.org/", "name": "US Center Safe Sport"}
]

FINANCE_SITES = [
    {"url": "https://www.nyse.com/index", "name": "NYSE USA"},
    {"url": "https://www.nasdaq.com/", "name": "NASDAQ USA"},
    {"url": "https://www.cmegroup.com/", "name": "CME Group USA"},
    {"url": "https://www.cboe.com/", "name": "CBOE USA"},
    {"url": "https://www.nadex.com/", "name": "NADEX USA"},
    {"url": "https://www.tmx.com/", "name": "TMX Canada"},
    {"url": "https://www.m-x.ca/en/", "name": "MX Canada"},
    {"url": "https://www.bcr.com.ar/eng/default.aspx", "name": "BCR Argentina"},
    {"url": "https://www.euronext.com/en", "name": "Euronext Europe"},
    {"url": "https://www.eurex.com/ex-en/", "name": "Eurex Germany"},
    {"url": "https://deutsche-boerse.com/dbg-en/", "name": "Deutsche Boerse"},
    {"url": "https://www.lseg.com/en", "name": "LSEG London"},
    {"url": "https://www.lme.com/", "name": "LME London"},
    {"url": "https://www.six-group.com/en/", "name": "SIX Swiss Exchange"},
    {"url": "https://www.nordpoolgroup.com/", "name": "Nord Pool Energy"},
    {"url": "https://www.epexspot.com/en", "name": "EPEX Spot Energy"},
    {"url": "https://www.bolsasymercados.es/ing/Home", "name": "BME Spain"},
    {"url": "https://www.dgcx.ae/", "name": "DGCX Dubai"},
    {"url": "https://www.gulfmerc.com/", "name": "Gulf Mercantile"},
    {"url": "https://www.nasdaqdubai.com/", "name": "NASDAQ Dubai"},
    {"url": "https://www.jse.co.za/trade", "name": "JSE South Africa"},
    {"url": "https://www.ea-africaexchange.com/", "name": "EA Africa Exchange"},
    {"url": "https://www.ecx.com.et/", "name": "ECX Ethiopia"},
    {"url": "https://www.hkex.com.hk/", "name": "HKEX Hong Kong"},
    {"url": "https://www.jpx.co.jp/english/", "name": "JPX Japan"},
    {"url": "https://www.tfx.co.jp/en/", "name": "TFX Japan"},
    {"url": "https://www.sse.com.cn/", "name": "SSE Shanghai"},
    {"url": "https://www.szse.cn/English/", "name": "SZSE Shenzhen"},
    {"url": "http://www.cffex.com.cn/", "name": "CFFEX China"},
    {"url": "http://www.dce.com.cn/", "name": "DCE Dalian China"},
    {"url": "https://www.bseindia.com/", "name": "BSE India"},
    {"url": "https://www.nseindia.com/", "name": "NSE India"},
    {"url": "https://www.mcxindia.com/", "name": "MCX India"},
    {"url": "https://global.krx.co.kr/", "name": "KRX Korea"},
    {"url": "https://www.sgx.com/", "name": "SGX Singapore"},
    {"url": "https://www.twse.com.tw/en/", "name": "TWSE Taiwan"},
    {"url": "https://mce.mn/", "name": "MCE Mongolia"},
    {"url": "https://www.miaxglobal.com/", "name": "MIAX USA"},
    {"url": "http://www.shfe.com.cn/en/", "name": "SHFE Shanghai"},
    {"url": "http://www.czce.com.cn/en/", "name": "CZCE Zhengzhou"},
    {"url": "https://www.ncdex.com/markets/", "name": "NCDEX India"}
]

SITES_TO_SCAN = SPORTS_SITES + FINANCE_SITES

# ── 3. THE "LIVING VOICE" ENGINE (EZRA & JESSA) ────────────
def generate_stewardship_dialogue(intel_report, alerts):
    print("🎙️ Ezra and Jessa are synthesizing the Living Pulse...")

    if alerts:
        script = f"Ezra: Initializing 4-hour scan summary. {intel_report[:400]}... Jessa: Ezra, wait. We have a High Alert breakthrough: {alerts}. We must pivot our stewardship focus here."
    else:
        script = f"Ezra: 4-hour Stewardship scan complete. All sectors are stable. {intel_report[:800]}"

    try:
        response = client.models.generate_content(
            model="gemini-2.0-flash-exp",
            contents=script,
            config=types.GenerateContentConfig(
                system_instruction="Ezra (Speaker 1) is firm and informative. Jessa (Speaker 2) is warm and wise. Jessa interrupts urgently if alerts exist.",
                speech_config=types.SpeechConfig(
                    multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
                        speaker_voice_configs=[
                            types.SpeakerVoiceConfig(speaker='Ezra', voice_config=types.VoiceConfig(prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name='Kore'))),
                            types.SpeakerVoiceConfig(speaker='Jessa', voice_config=types.VoiceConfig(prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name='Sulafat')))
                        ]
                    )
                )
            )
        )
        filename = f"Pulse_{datetime.datetime.now().strftime('%H%M')}.mp3"
        with open(filename, "wb") as f:
            f.write(response.audio_bytes)
        return filename
    except Exception as e:
        print(f"⚠️ Audio Error: {e}")
        return None

# ── 4. LOGISTICS & DELIVERY ──────────────────────────
def sync_and_purge(file_path):
    service = build('drive', 'v3')
    cutoff = (datetime.datetime.now() - datetime.timedelta(days=60)).isoformat() + 'Z'
    try:
        old_files = service.files().list(q=f"'{DRIVE_FOLDER_ID}' in parents and createdTime < '{cutoff}'").execute()
        for f in old_files.get('files', []): service.files().delete(fileId=f['id']).execute()
    except: pass

    meta = {'name': os.path.basename(file_path), 'parents': [DRIVE_FOLDER_ID]}
    media = MediaFileUpload(file_path, mimetype='audio/mpeg')
    file = service.files().create(body=meta, media_body=media, fields='id, webViewLink').execute()
    service.permissions().create(fileId=file.get('id'), body={'type': 'anyone', 'role': 'reader'}).execute()
    return file.get('webViewLink')

def deliver_email(report_text, link, alert_count):
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"🛡️ Pulse Update: {alert_count} Alerts Detected"
    msg["From"], msg["To"] = SENDER_EMAIL, ", ".join(CONTACT_EMAILS)

    html = f"""<div style="font-family:Arial; padding:20px; border:1px solid #eee;">
    <h2 style="color:#1a1a2e;">Stewardship Pulse</h2>
    <p><a href="{link}" style="background:#cc0000; color:white; padding:10px 20px; text-decoration:none; border-radius:5px;">▶ LISTEN TO EZRA & JESSA</a></p>
    <hr><p>{report_text[:1000]}...</p>
    <p style="font-size:10px; color:gray;">Signed Management | God Bless</p>
    </div>"""
    msg.attach(MIMEText(html, "html"))

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.sendmail(SENDER_EMAIL, CONTACT_EMAILS, msg.as_string())

# ── 5. THE PIPELINE ──────────────────────────────────
def execute_pulse_cycle():
    print(f"\n🔄 CYCLE START — {datetime.datetime.now(OK_TZ)}")
    scraped_text, high_alerts = "", []

    for site in tqdm(SITES_TO_SCAN, desc="🔎 Ezra & Jessa Scanning"):
        try:
            res = requests.get(site["url"], timeout=8, headers={"User-Agent":"Mozilla/5.0"})
            content = BeautifulSoup(res.text, "html.parser").get_text()[:500]
            scraped_text += f"\n---\n{site['name']}: {content}"
            if any(x in content.lower() for x in ["opportunity", "breakthrough", "efficiency"]):
                high_alerts.append(f"Breakthrough at {site['name']}")
        except: continue

    alert_summary = " | ".join(high_alerts) if high_alerts else None
    audio_path = generate_stewardship_dialogue(scraped_text, alert_summary)

    if audio_path:
        link = sync_and_purge(audio_path)
        deliver_email(scraped_text, link, len(high_alerts))
        print(f"📧 Pulse Delivered at {datetime.datetime.now(OK_TZ)}")

# ── 6. IGNITION ──────────────────────────────────────
if __name__ == "__main__":
    while True:
        execute_pulse_cycle()
        print("💤 Resting 4 hours...")
        time.sleep(14400)